Estratégias para manipulação de dados

## Preparação — Dados de pedidos

A tabela `orders` contém identificador, situação, data, prioridade e valor do pedido. Os registros são disponibilizados pelo Databricks no catálogo `samples`.

**Caso de uso:** uma distribuidora deseja transformar registros de pedidos em indicadores comerciais.

In [ ]:
bronze = (
    spark.table("samples.tpch.orders")
    .select(
        "o_orderkey", "o_orderstatus", "o_totalprice",
        "o_orderdate", "o_orderpriority"
    )
    .limit(20000)
)

## Ciclo 1 — Bronze: conhecer os dados

**Demonstração:** leitura da tabela, inspeção do esquema, seleção de colunas, contagem e visualização. A Bronze preserva os valores selecionados da fonte.

In [ ]:
bronze.printSchema()
display(bronze.select("o_orderkey", "o_orderstatus", "o_totalprice").limit(6))
print("Quantidade de pedidos:", bronze.count())

**Atividade 1 — Faça você:** usando `bronze`, selecione `o_orderkey`, `o_orderdate` e `o_orderpriority`, mostre **6 registros** e imprima a quantidade de pedidos. Repita o procedimento demonstrado.

In [ ]:
# TODO — Atividade 1
# Selecione as colunas solicitadas, use limit(), display() e count().

## Ciclo 2 — Silver: limpeza e validação

**Demonstração:** identificar possíveis valores inválidos e manter pedidos com identificador, situação e data preenchidos e valor positivo. Também removemos identificadores duplicados.

**Atenção:** o conjunto de exemplo pode não conter registros problemáticos; uma boa regra de qualidade é necessária mesmo quando nenhuma linha é descartada.

In [ ]:
from pyspark.sql.functions import col

silver = (
    bronze
    .dropDuplicates(["o_orderkey"])
    .filter(col("o_orderkey").isNotNull())
    .filter(col("o_orderstatus").isNotNull())
    .filter(col("o_orderdate").isNotNull())
    .filter(col("o_totalprice") > 0)
)

print("Registros na Bronze:", bronze.count())
print("Registros na Silver:", silver.count())
display(silver.limit(6))

# Demonstração: filtre a Silver por uma situação de pedido.
pedidos_fechados = silver.filter(col("o_orderstatus") == "F")
display(pedidos_fechados.limit(5))
print("Pedidos com situação F:", pedidos_fechados.count())

**Atividade 2 — Faça você:** crie `pedidos_abertos` usando a Silver e o filtro `o_orderstatus == "O"`. Exiba até 5 registros e informe a contagem. Utilize `filter()`, `display()` e `count()` como demonstrado.

In [ ]:
# TODO — Atividade 2
# Crie pedidos_abertos, exiba até 5 registros e conte.

## Ciclo 3 — Silver: transformação e enriquecimento

**Demonstração:** criar o ano de cada pedido e uma categoria de valor com `withColumn()`, `year()` e `when()`. Para esta atividade, pedidos a partir de 100.000 unidades monetárias serão classificados como **alto valor**.

O limite é uma **regra didática**, não uma regra oficial do TPC-H.

In [ ]:
from pyspark.sql.functions import year, when

silver_enriquecida = (
    silver
    .withColumn("ano_pedido", year(col("o_orderdate")))
    .withColumn(
        "faixa_valor",
        when(col("o_totalprice") >= 100000, "Alto").otherwise("Regular")
    )
)

display(
    silver_enriquecida.select(
        "o_orderkey", "o_orderdate", "ano_pedido",
        "o_totalprice", "faixa_valor"
    ).limit(8)
)

**Atividade 3 — Faça você:** a partir de `silver_enriquecida`, crie `silver_prioridades`, acrescentando a coluna `pedido_urgente`: **"Sim"** se `o_orderpriority == "1-URGENT"`, ou **"Não"** caso contrário. Exiba o identificador, a prioridade e a nova coluna.

In [ ]:
# TODO — Atividade 3
# Use withColumn() e when(). Caso contrário, use otherwise("Não").

## Ciclo 4 — Gold: agregações e indicadores

**Demonstração:** gerar indicadores por situação de pedido. Calcule total de pedidos, valor total e valor médio. A camada Gold entrega informação pronta para consumo.

In [ ]:
from pyspark.sql.functions import count, sum as sum_, avg, round

gold_status = (
    silver_enriquecida
    .groupBy("o_orderstatus")
    .agg(
        count("*").alias("quantidade_pedidos"),
        round(sum_("o_totalprice"), 2).alias("valor_total"),
        round(avg("o_totalprice"), 2).alias("valor_medio")
    )
    .orderBy(col("valor_total").desc())
)

display(gold_status)

**Atividade 4 — Faça você:** crie `gold_prioridade` a partir de `silver_enriquecida`, repetindo as **mesmas três métricas**, mas agrupando por `o_orderpriority`. Ordene pelo maior valor total e exiba a tabela.

In [ ]:
# TODO — Atividade 4
# Repita groupBy(), agg(), orderBy() e display(), agora por prioridade.

## Desafio final — Quais pedidos merecem atenção?

A equipe comercial quer descobrir **quais combinações de situação e prioridade concentram maior valor de pedidos**.

Usando `silver_enriquecida`, produza `gold_status_prioridade`:

1. Agrupe por `o_orderstatus` e `o_orderpriority`.
2. Calcule `quantidade_pedidos`, `valor_total` e `valor_medio` conforme a demonstração.
3. Ordene por `valor_total` decrescente.
4. Exiba o resultado e explique qual grupo aparece primeiro.

**Apenas operações demonstradas durante os ciclos são necessárias.**

In [ ]:
# TODO — Desafio final
# Crie e exiba gold_status_prioridade.